In [4]:
import pandas as pd
import numpy as np
from scipy import stats
import pyterrier as pt
if not pt.started():
    pt.init()

# 设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)
pd.set_option('display.precision', 4)

print("=" * 100)
print("QPP CORRELATION ANALYSIS FOR PRF EFFECTIVENESS PREDICTION")
print("=" * 100)

# ========== CONFIGURABLE PARAMETERS ==========
MIN_B_PREF_RATIO = 0.0
print(f"\n📊 Configuration:")
print(f"   Including queries with b_preference_ratio > {MIN_B_PREF_RATIO}")
print("=" * 100)

# ========== 1. LOAD DATA ==========
print("\n1. Loading Retrieval Results...")
print("-" * 100)

# Read as a standard CSV file
df_colbert = pd.read_csv("df_colbert_deduped.csv")
df_colbert_prf = pd.read_csv("df_colbert_prf_deduped.csv")

# ===== 新增：诊断 CSV 文件结构 =====
print("\n🔍 DIAGNOSTIC: Checking CSV file structure...")
print(f"\ndf_colbert shape: {df_colbert.shape}")
print(f"df_colbert columns: {df_colbert.columns.tolist()}")
print(f"df_colbert first 3 rows:\n{df_colbert.head(3)}")
print(f"\ndf_colbert_prf shape: {df_colbert_prf.shape}")
print(f"df_colbert_prf columns: {df_colbert_prf.columns.tolist()}")
print(f"df_colbert_prf first 3 rows:\n{df_colbert_prf.head(3)}")

# ===== 修复：确保所有必需的列存在 =====
def fix_dataframe_columns(df, df_name):
    """修复 DataFrame 的列名和数据类型"""
    print(f"\n🔧 Fixing {df_name}...")
    
    # 检查并重命名列
    expected_cols = ['qid', 'docno', 'score', 'rank']
    
    # 如果列数不对，尝试推断
    if len(df.columns) < len(expected_cols):
        print(f"  ⚠️ Warning: {df_name} has only {len(df.columns)} columns")
        # 根据列数推断列名
        if len(df.columns) == 3:
            df.columns = ['qid', 'docno', 'score']
        elif len(df.columns) == 4:
            df.columns = ['qid', 'docno', 'score', 'rank']
        else:
            print(f"  ❌ Error: Unexpected column count: {len(df.columns)}")
            print(f"     Columns: {df.columns.tolist()}")
            return None
    
    # 确保 qid 和 docno 是字符串
    if 'qid' in df.columns:
        df['qid'] = df['qid'].astype(str).str.strip()
    if 'docno' in df.columns:
        df['docno'] = df['docno'].astype(str).str.strip()
    
    # 确保 score 是数值
    if 'score' in df.columns:
        df['score'] = pd.to_numeric(df['score'], errors='coerce')
    
    # 如果没有 rank 列，创建一个
    if 'rank' not in df.columns:
        df['rank'] = df.groupby('qid').cumcount() + 1
        print(f"  ✓ Created rank column")
    
    print(f"  ✓ Fixed columns: {df.columns.tolist()}")
    print(f"  ✓ qid dtype: {df['qid'].dtype}")
    print(f"  ✓ Sample qids: {df['qid'].head(3).tolist()}")
    
    return df

df_colbert = fix_dataframe_columns(df_colbert, "df_colbert")
df_colbert_prf = fix_dataframe_columns(df_colbert_prf, "df_colbert_prf")

# 检查修复结果
if df_colbert is None or df_colbert_prf is None:
    print("\n❌ Error: Cannot fix CSV structure. Please check your files.")
    exit(1)

print(f"\n✓ ColBERT E2E: {len(df_colbert)} results, {df_colbert['qid'].nunique()} queries")
print(f"✓ ColBERT PRF: {len(df_colbert_prf)} results, {df_colbert_prf['qid'].nunique()} queries")

# Load preference data
try:
    preference_df = pd.read_csv('preference.csv')
    preference_df['qid'] = preference_df['qid'].astype(str).str.strip()
    print(f"✓ Preference data: {len(preference_df)} queries")
    print(f"  Sample preference qids: {preference_df['qid'].head(3).tolist()}")
except FileNotFoundError:
    print("ERROR: preference.csv not found")
    exit(1)

# Load Dense QPP (optional)
dense_qpp_df = None
try:
    dense_qpp_df = pd.read_csv('rbo_Noisyq_0.01.dl2019_queries.tsv.tsv_500', sep='\t')
    if len(dense_qpp_df.columns) >= 2:
        dense_qpp_df = dense_qpp_df.iloc[:, :2]
        dense_qpp_df.columns = ['qid', 'dense_qpp']
        dense_qpp_df['qid'] = dense_qpp_df['qid'].astype(str).str.strip()
        print(f"✓ Dense QPP: {len(dense_qpp_df)} queries")
except:
    print("  Note: Dense QPP file not found, skipping")

# ========== 2. CALCULATE QPP SCORES ==========
print("\n2. Calculating QPP Scores...")
print("-" * 100)

def calculate_nqc(df, group_by='qid', score_col='score'):
    """NQC = std(scores) / mean(scores) - measures score divergence"""
    results = []
    for qid, group in df.groupby(group_by):
        scores = group[score_col].values
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        nqc = std_score / mean_score if mean_score != 0 else 0
        results.append({'qid': str(qid), 'nqc': nqc})
    return pd.DataFrame(results)

def calculate_wig(df_baseline, df_prf, group_by='qid', score_col='score', top_k=10):
    """WIG = mean(PRF_scores) - mean(baseline_scores) - measures score improvement"""
    results = []
    for qid in df_baseline[group_by].unique():
        baseline_group = df_baseline[df_baseline[group_by] == qid].head(top_k)
        prf_group = df_prf[df_prf[group_by] == qid].head(top_k)
        
        if len(baseline_group) == 0 or len(prf_group) == 0:
            continue
        
        baseline_mean = baseline_group[score_col].mean()
        prf_mean = prf_group[score_col].mean()
        wig = prf_mean - baseline_mean
        results.append({'qid': str(qid), 'wig': wig})
    return pd.DataFrame(results)

def calculate_uef(df, group_by='qid', score_col='score', top_k=10):
    """UEF = mean(scores) × (1 - std(scores)) - combines mean and consistency"""
    results = []
    for qid, group in df.groupby(group_by):
        top_scores = group.head(top_k)[score_col].values
        if len(top_scores) == 0:
            continue
        mean_score = np.mean(top_scores)
        std_score = np.std(top_scores)
        uef = mean_score * (1 - std_score) if std_score < 1 else 0
        results.append({'qid': str(qid), 'uef': uef})
    return pd.DataFrame(results)

def calculate_smv(df, group_by='qid', score_col='score', top_k=10):
    """SMV = std(scores) / sqrt(k) - normalized score variance"""
    results = []
    for qid, group in df.groupby(group_by):
        top_scores = group.head(top_k)[score_col].values
        if len(top_scores) == 0:
            continue
        std_score = np.std(top_scores)
        smv = std_score / np.sqrt(len(top_scores))
        results.append({'qid': str(qid), 'smv': smv})
    return pd.DataFrame(results)

print("  Calculating NQC...")
nqc_df = calculate_nqc(df_colbert)
print(f"    ✓ {len(nqc_df)} queries")
print(f"    Sample NQC qids: {nqc_df['qid'].head(3).tolist()}")

print("  Calculating WIG...")
wig_df = calculate_wig(df_colbert, df_colbert_prf)
print(f"    ✓ {len(wig_df)} queries")
print(f"    Sample WIG qids: {wig_df['qid'].head(3).tolist()}")

print("  Calculating UEF...")
uef_df = calculate_uef(df_colbert)
print(f"    ✓ {len(uef_df)} queries")
print(f"    Sample UEF qids: {uef_df['qid'].head(3).tolist()}")

print("  Calculating SMV...")
smv_df = calculate_smv(df_colbert)
print(f"    ✓ {len(smv_df)} queries")
print(f"    Sample SMV qids: {smv_df['qid'].head(3).tolist()}")

# Save individual QPP scores
nqc_df.to_csv('nqc_scores.csv', index=False)
wig_df.to_csv('wig_scores.csv', index=False)
uef_df.to_csv('uef_scores.csv', index=False)
smv_df.to_csv('smv_scores.csv', index=False)
print("\n  ✓ Individual QPP scores saved")

# ========== 3. MERGE DATA ==========
print("\n3. Merging Data...")
print("-" * 100)

# ===== 新增：检查 qid 格式一致性 =====
print("\n🔍 Checking qid consistency before merge...")
print(f"preference_df qid sample: {preference_df['qid'].head(3).tolist()}")
print(f"nqc_df qid sample: {nqc_df['qid'].head(3).tolist()}")
print(f"wig_df qid sample: {wig_df['qid'].head(3).tolist()}")

# 确保所有 qid 都是字符串且去除空格
preference_df['qid'] = preference_df['qid'].astype(str).str.strip()
nqc_df['qid'] = nqc_df['qid'].astype(str).str.strip()
wig_df['qid'] = wig_df['qid'].astype(str).str.strip()
uef_df['qid'] = uef_df['qid'].astype(str).str.strip()
smv_df['qid'] = smv_df['qid'].astype(str).str.strip()

merged_df = preference_df[['qid', 'b_preference_ratio']].copy()
print(f"  Starting with {len(merged_df)} queries from preference.csv")

merged_df = pd.merge(merged_df, nqc_df, on='qid', how='left')
print(f"  After NQC merge: {len(merged_df)} rows, {merged_df['nqc'].notna().sum()} with NQC scores")

merged_df = pd.merge(merged_df, wig_df, on='qid', how='left')
print(f"  After WIG merge: {len(merged_df)} rows, {merged_df['wig'].notna().sum()} with WIG scores")

merged_df = pd.merge(merged_df, uef_df, on='qid', how='left')
print(f"  After UEF merge: {len(merged_df)} rows, {merged_df['uef'].notna().sum()} with UEF scores")

merged_df = pd.merge(merged_df, smv_df, on='qid', how='left')
print(f"  After SMV merge: {len(merged_df)} rows, {merged_df['smv'].notna().sum()} with SMV scores")

if dense_qpp_df is not None:
    dense_qpp_df['qid'] = dense_qpp_df['qid'].astype(str).str.strip()
    merged_df = pd.merge(merged_df, dense_qpp_df, on='qid', how='left')
    print(f"  After Dense QPP merge: {len(merged_df)} rows")

# Filter: remove queries with insufficient data
merged_df = merged_df[merged_df['b_preference_ratio'] > MIN_B_PREF_RATIO].copy()
print(f"  After filtering: {len(merged_df)} queries with valid b_preference_ratio")

# Check data availability
qpp_columns = ['nqc', 'wig', 'uef', 'smv']
if dense_qpp_df is not None:
    qpp_columns.append('dense_qpp')

print("\n  QPP Score Availability:")
available_qpps = []
for col in qpp_columns:
    if col in merged_df.columns:
        valid_count = merged_df[col].notna().sum()
        coverage = valid_count / len(merged_df) * 100
        print(f"    {col.upper():12} {valid_count:3}/{len(merged_df)} ({coverage:5.1f}%)")
        if valid_count >= 10:  # Need at least 10 samples for meaningful correlation
            available_qpps.append(col)

print(f"\n  QPP methods with sufficient data: {', '.join([q.upper() for q in available_qpps])}")

# Save merged data
merged_df.to_csv('qpp_merged_data.csv', index=False)
print("  ✓ Merged data saved: qpp_merged_data.csv")

# ========== 4. CORRELATION ANALYSIS ==========
print("\n" + "=" * 100)
print("4. CORRELATION ANALYSIS: QPP Scores vs PRF Effectiveness (b_preference_ratio)")
print("=" * 100)

print("""
INTERPRETATION GUIDE:
• b_preference_ratio > 0.5 → PRF performs better
• b_preference_ratio < 0.5 → Baseline performs better
• Correlation coefficient ∈ [-1, 1]:
  - Positive: Higher QPP score → PRF more beneficial
  - Negative: Lower QPP score → PRF more beneficial
  - Close to 0: No relationship between QPP and PRF effectiveness
""")

correlation_results = []

for qpp_col in available_qpps:
    # Get valid pairs (remove NaN)
    valid_data = merged_df[[qpp_col, 'b_preference_ratio']].dropna()
    
    if len(valid_data) < 10:
        print(f"\n{qpp_col.upper()}: Insufficient data (n={len(valid_data)}), skipping")
        continue
    
    qpp_scores = valid_data[qpp_col].values
    b_pref_ratios = valid_data['b_preference_ratio'].values
    
    # Kendall's Tau (rank correlation, robust to outliers)
    tau, p_tau = stats.kendalltau(qpp_scores, b_pref_ratios)
    
    # Spearman's Rho (rank correlation, monotonic relationships)
    rho, p_rho = stats.spearmanr(qpp_scores, b_pref_ratios)
    
    # Pearson's r (linear correlation)
    r, p_r = stats.pearsonr(qpp_scores, b_pref_ratios)
    
    # Determine significance level
    def sig_marker(p):
        if p < 0.001:
            return "***"
        elif p < 0.01:
            return "**"
        elif p < 0.05:
            return "*"
        else:
            return ""
    
    # Interpret correlation strength
    def interpret_strength(coef):
        abs_coef = abs(coef)
        if abs_coef < 0.1:
            return "Negligible"
        elif abs_coef < 0.3:
            return "Weak"
        elif abs_coef < 0.5:
            return "Moderate"
        elif abs_coef < 0.7:
            return "Strong"
        else:
            return "Very Strong"
    
    # Interpret direction
    direction = "Positive" if tau > 0 else "Negative" if tau < 0 else "None"
    
    correlation_results.append({
        'QPP_Method': qpp_col.upper(),
        'N_Samples': len(valid_data),
        'Kendall_Tau': tau,
        'Tau_P_Value': p_tau,
        'Tau_Sig': sig_marker(p_tau),
        'Spearman_Rho': rho,
        'Rho_P_Value': p_rho,
        'Rho_Sig': sig_marker(p_rho),
        'Pearson_R': r,
        'R_P_Value': p_r,
        'R_Sig': sig_marker(p_r),
        'Strength': interpret_strength(tau),
        'Direction': direction
    })

# Create results DataFrame
results_df = pd.DataFrame(correlation_results)

# ========== 5. DISPLAY RESULTS ==========
print("\n" + "-" * 100)
print("TABLE 1: Correlation Coefficients")
print("-" * 100)

# Main table
display_df = results_df[['QPP_Method', 'N_Samples', 'Kendall_Tau', 'Tau_Sig', 
                          'Spearman_Rho', 'Rho_Sig', 'Strength', 'Direction']].copy()
display_df.columns = ['Method', 'N', 'Kendall τ', '', 'Spearman ρ', '', 'Strength', 'Direction']
print(display_df.to_string(index=False))

print("\n" + "-" * 100)
print("TABLE 2: Statistical Significance (P-values)")
print("-" * 100)

pvalue_df = results_df[['QPP_Method', 'Tau_P_Value', 'Rho_P_Value', 'R_P_Value']].copy()
pvalue_df.columns = ['Method', 'Kendall p', 'Spearman p', 'Pearson p']
print(pvalue_df.to_string(index=False))

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SIGNIFICANCE MARKERS: * p<0.05, ** p<0.01, *** p<0.001
STRENGTH CATEGORIES: Negligible (<0.1), Weak (0.1-0.3), Moderate (0.3-0.5), Strong (0.5-0.7), Very Strong (≥0.7)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# Save results
results_df.to_csv('qpp_correlation_analysis.csv', index=False)
print("✓ Correlation results saved: qpp_correlation_analysis.csv")

# ========== 6. DETAILED INTERPRETATION ==========
print("\n" + "=" * 100)
print("5. DETAILED INTERPRETATION")
print("=" * 100)

for _, row in results_df.iterrows():
    method = row['QPP_Method']
    tau = row['Kendall_Tau']
    p_tau = row['Tau_P_Value']
    strength = row['Strength']
    direction = row['Direction']
    n = row['N_Samples']
    
    print(f"\n{method}:")
    print(f"  Sample Size: {n}")
    print(f"  Kendall's τ = {tau:.4f} (p = {p_tau:.4f})")
    print(f"  Correlation: {strength} {direction}")
    
    if p_tau < 0.05:
        print(f"  ✓ SIGNIFICANT correlation detected")
        if tau > 0:
            print(f"    → Higher {method} scores predict BETTER PRF effectiveness")
            print(f"    → Interpretation: Queries with high {method} benefit more from PRF")
        else:
            print(f"    → Lower {method} scores predict BETTER PRF effectiveness")
            print(f"    → Interpretation: Queries with low {method} benefit more from PRF")
    else:
        print(f"  ✗ NO significant correlation (p ≥ 0.05)")
        print(f"    → {method} cannot reliably predict PRF effectiveness")
        print(f"    → PRF benefit appears independent of this query characteristic")

# ========== 7. SUMMARY & RECOMMENDATIONS ==========
print("\n" + "=" * 100)
print("6. SUMMARY & RECOMMENDATIONS")
print("=" * 100)

significant_methods = results_df[results_df['Tau_P_Value'] < 0.05]
best_method = results_df.loc[results_df['Kendall_Tau'].abs().idxmax()] if len(results_df) > 0 else None

print(f"\n📊 Overall Findings:")
print(f"   Total QPP methods evaluated: {len(results_df)}")
print(f"   Methods with significant correlation: {len(significant_methods)}")

if len(significant_methods) > 0:
    print(f"\n🏆 Significant Predictors:")
    for _, row in significant_methods.iterrows():
        print(f"   • {row['QPP_Method']:12} τ={row['Kendall_Tau']:6.3f} ({row['Strength']} {row['Direction']})")
    
    print(f"\n   Best predictor: {best_method['QPP_Method']} (|τ|={abs(best_method['Kendall_Tau']):.3f})")
else:
    print(f"\n⚠️  NO QPP methods showed significant correlation with PRF effectiveness")

print(f"\n💡 Implications for Your Research:")

if len(significant_methods) == 0:
    print("""
   ✓ GOOD NEWS for your research!
     → Simple QPP scores CANNOT predict when PRF helps
     → This validates the need for your sophisticated selective PRF approach
     → Your method has the opportunity to significantly outperform these baselines
     
   📝 For Your Paper:
     → Report these weak correlations as motivation
     → Argue that PRF effectiveness depends on factors beyond simple query statistics
     → Position your method as addressing this limitation
     
   🎯 Expected Performance:
     → Even modest improvement (Kendall τ > 0.3) would be meaningful
     → Focus on interpretability and practical applicability
""")
elif abs(best_method['Kendall_Tau']) < 0.3:
    print("""
   → Weak correlations detected, but not strong predictors
   → Your method still has room to significantly improve
   → Target: Kendall τ > 0.4 for meaningful advancement
""")
elif abs(best_method['Kendall_Tau']) < 0.5:
    print("""
   → Moderate correlations exist
   → Your method needs to substantially outperform these baselines
   → Target: Kendall τ > 0.5 for significant contribution
   → Consider combining multiple QPP signals
""")
else:
    print("""
   ⚠️  Strong QPP predictors detected
   → This is a challenging baseline to beat
   → Your method must show clear advantages
   → Consider: different evaluation scenarios, efficiency, interpretability
""")

print(f"\n📁 Output Files:")
print(f"   ✓ qpp_correlation_analysis.csv - Main correlation results")
print(f"   ✓ qpp_merged_data.csv - Full dataset for further analysis")
print(f"   ✓ nqc_scores.csv, wig_scores.csv, uef_scores.csv, smv_scores.csv")

print("\n" + "=" * 100)
print("ANALYSIS COMPLETE!")
print("=" * 100)

# ========== 8. GENERATE LATEX TABLE (BONUS) ==========
print("\n" + "=" * 100)
print("7. LATEX TABLE FOR PAPER")
print("=" * 100)

print("""
\\begin{table}[t]
\\centering
\\caption{Correlation between QPP methods and PRF effectiveness (b\\_preference\\_ratio)}
\\label{tab:qpp_correlation}
\\begin{tabular}{lcccc}
\\hline
\\textbf{Method} & \\textbf{N} & \\textbf{Kendall's $\\tau$} & \\textbf{Spearman's $\\rho$} & \\textbf{Strength} \\\\
\\hline
""")

for _, row in results_df.iterrows():
    tau_str = f"{row['Kendall_Tau']:.3f}{row['Tau_Sig']}"
    rho_str = f"{row['Spearman_Rho']:.3f}{row['Rho_Sig']}"
    print(f"{row['QPP_Method']:12} & {row['N_Samples']:3} & {tau_str:10} & {rho_str:10} & {row['Strength']:12} \\\\")

print("""\\hline
\\multicolumn{5}{l}{\\footnotesize * $p<0.05$, ** $p<0.01$, *** $p<0.001$}
\\end{tabular}
\\end{table}
""")

print("\n✓ Copy the above LaTeX code directly into your paper!")

/tmp/ipykernel_4608/3930654908.py:5: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


QPP CORRELATION ANALYSIS FOR PRF EFFECTIVENESS PREDICTION

📊 Configuration:
   Including queries with b_preference_ratio > 0.0

1. Loading Retrieval Results...
----------------------------------------------------------------------------------------------------

🔍 DIAGNOSTIC: Checking CSV file structure...

df_colbert shape: (281534, 6)
df_colbert columns: ['qid', 'docno', 'rank', 'score', 'name', 'passage_text']
df_colbert first 3 rows:
       qid    docno  rank    score       name                                       passage_text
0  1037798  8760871     0  25.2152  pyterrier  Atlantic Ocean, United States. Robert Gray, (b...
1  1037798  7822415     1  23.7989  pyterrier  Captain Robert Gray carries the flag around th...
2  1037798  8760866     2  23.7095  pyterrier  Robert Gray was the Democratic candidate for g...

df_colbert_prf shape: (37996, 6)
df_colbert_prf columns: ['qid', 'docno', 'rank', 'score', 'name', 'passage_text']
df_colbert_prf first 3 rows:
       qid    docno  rank 